# GCS auto-annotate in ApertureDB: querying what the detectors found

Google Cloud Storage's **auto-annotate** runs detectors (labels, objects, faces, content safety) on the
images in a bucket. `ingest.py` and `enrich.py` copied the images and those detections into ApertureDB
as a graph; this notebook queries it.

```
 _Image ──HasLabel──▶ GCSLabel            50 whole-image labels with a score
   ▲
   └──── _BoundingBox                     object boxes (named by the object) and face boxes (label "face"),
                                          each with its own properties (score, joy, pan angle, ...)
```

Everything here is **phase 1**: structured metadata only, no embeddings or similarity search yet.
The queries use ApertureDB's string operators, property expressions and graph traversal. The same queries
are saved as JSON in `queries/` so you can paste them into the ApertureDB web UI.

Run `./setup.sh` and `./demo.sh --bucket ... --prefix ...` first (see the README).

## Connect

In [ ]:
import html
import json
import os

import pandas as pd
from IPython.display import HTML, display

import config          # which ApertureDB: local Community Edition, or APERTUREDB_KEY / .env
import graph_viz as gv  # interactive graph views
import queries         # every query below, as plain functions returning ApertureDB queries

client = config.get_client()

DATASET = None  # set to "<bucket>/<prefix>" to restrict every query to one ingested folder
LABELS = {"_Image": "gcs_object_stem", "GCSLabel": "description"}   # how graph nodes are named
BY_SCORE = {"GCSLabel": ("score", "descending")}                    # keep the best labels first

In [ ]:
def run(query):
    """Run a query; raise if the server rejects a command; return (response, blobs)."""
    response, blobs = client.query(query)
    if isinstance(response, dict):
        raise RuntimeError(response.get("info"))
    for r in response:
        body = next(iter(r.values()))
        if body.get("status", 0) not in (0, 2):
            raise RuntimeError(body.get("info"))
    return response, blobs


def table(response, index=-1, columns=None):
    """The entities one command returned, as a DataFrame."""
    rows = next(iter(response[index].values())).get("entities", [])
    if isinstance(rows, dict):  # grouped by source
        rows = [e for group in rows.values() for e in group]
    df = pd.DataFrame(rows)
    return df if columns is None or df.empty else df[[c for c in columns if c in df]]


def thumbs(blobs, captions, width=170):
    """Show returned pixels (posters or box crops) in a row."""
    import base64
    cells = "".join(
        f'<figure style="display:inline-block;margin:4px;text-align:center;font:12px sans-serif">'
        f'<img width="{width}" src="data:{gv.sniff_mime(b)};base64,{base64.b64encode(b).decode()}">'
        f'<figcaption>{html.escape(str(c))}</figcaption></figure>' for b, c in zip(blobs, captions))
    display(HTML(cells))


def show(query, caption=None, columns=None):
    """Show the query (collapsed), run it, show the rows and - if it returned pixels - the pictures."""
    display(HTML("<details><summary>the query</summary><pre>" + html.escape(json.dumps(query, indent=1)) + "</pre></details>"))
    response, blobs = run(query)
    df = table(response, columns=columns)
    display(df if len(df) else HTML("<i>no rows - ingest more data or loosen the filter</i>"))
    if blobs and caption and len(df):
        thumbs(blobs, [caption(r) for _, r in df.iterrows()])
    return response, blobs

## The data model

Live from the database: one node per class (with how many objects it holds), one edge per connection class.
Hover a class to highlight its connections, click it for its properties and a query that lists it.

In [ ]:
gv.render_schema(client)

## String search

Case-insensitive substring search (`icontains`) over label text, then traverse from the matching labels to
their posters. Trigram indexes on these properties keep it fast. Other operators: `contains`, `starts_with`,
`ends_with`, `like` (`%` and `_`), `regex` (RE2, `(?i)` for any case) and the case-insensitive `i...` versions.

In [ ]:
_ = show(queries.labels_containing("action", min_score=0.6, dataset=DATASET, images=True, limit=8),
         caption=lambda r: r.gcs_object_name.rsplit("/", 1)[-1])

A regular expression over object names (the object detector calls people man, woman, girl, boy, person ...):

In [ ]:
_ = show(queries.people_by_name(dataset=DATASET, limit=10))

Boolean text search in one request: any of some words, none of others, above a score, best first (try your own words, e.g. `golf` or `romance`).

In [ ]:
_ = show(queries.label_search_boolean(include=("action", "fiction"), exclude=("game",), dataset=DATASET),
         columns=["description", "score", "image_id"])

Patterns: labels that end in *ing* (`like '%ing'`) or start with *for* (`istarts_with`):

In [ ]:
_ = show(queries.label_shapes(dataset=DATASET), columns=['description', 'score'])

## Expressions: compare and compute on properties, inside the database

Faces looking at the camera (`abs(pan_angle) < 20`) that show more joy than sorrow *and* more joy than anger.
`$joy > $sorrow` compares two properties of the same face; nothing is post-processed in Python.
Because the faces are stored with their pixels, the request can return the cropped faces themselves.

In [ ]:
_ = show(queries.happy_frontal_faces(dataset=DATASET, crops=True, limit=8),
         caption=lambda r: f"joy {r.joy:.2f}", columns=["image_id", "joy", "pan_angle", "area_fraction"])

Arithmetic over properties (`joy + surprise > 0.85` while `sorrow < 0.1`):

In [ ]:
_ = show(queries.delighted_faces(dataset=DATASET))

Posters dominated by a single, confidently detected face (`area_fraction` is the share of the poster the face covers):

In [ ]:
_ = show(queries.dominant_faces(dataset=DATASET, crops=True),
         caption=lambda r: f"{r.area_fraction:.0%} of poster", columns=["image_id", "area_fraction", "detection_confidence"])

Content moderation: adult *or* violent content at least POSSIBLE (ranks 3 of 5), across two detectors' fields:

In [ ]:
_ = show(queries.sensitive_content(dataset=DATASET))

`is_missing` as a data-quality check: how many images have no face-detector result stored yet?

In [ ]:
response, _ = run(queries.not_yet_enriched(dataset=DATASET))
response[0]["FindImage"]["count"]

## Graph traversal

Three hops in one request: **label -> poster -> that poster's happy faces**.

In [ ]:
_ = show(queries.happy_faces_on_tagged_posters(term="film", min_joy=0.3, dataset=DATASET),
         columns=["image_id", "joy", "area_fraction"])

Posters connected to a label **and** to an object box (`is_connected_to` with `all`):

In [ ]:
_ = show(queries.label_and_object_together("clothing", "person", dataset=DATASET))

Counts stored at enrichment time make "crowded" posters a plain filter:

In [ ]:
_ = show(queries.crowded_posters(min_faces=3, min_objects=5, dataset=DATASET))

Flagged posters together with the labels that explain them:

In [ ]:
_ = show(queries.why_flagged(dataset=DATASET), columns=['description', 'score', 'image_id'])

### See it as a graph

Draw what a query returned, with its neighbours. Highlighted nodes are the answer; click any node for its
properties, its picture (with every stored box drawn over it) and the query that fetches it. Use the legend
to hide a class and the search box to find a node.

In [ ]:
gv.render_query(client, queries.happy_frontal_faces(dataset=DATASET, limit=5), title="happy frontal faces",
                label_props=LABELS, sort=BY_SCORE, per_class=4)

Start from one poster instead and walk outwards (`depth=2` also reaches the posters sharing its labels):

In [ ]:
response, _ = run(queries.crowded_posters(min_faces=1, dataset=DATASET, limit=1))
poster = table(response).gcs_object_name.iloc[0]
poster

In [ ]:
gv.render_neighborhood(client, "_Image", {"gcs_object_name": ["==", poster]}, depth=1, per_class=12,
                       label_props=LABELS, sort=BY_SCORE, title=poster)

## Spatial: boxes from two different detectors

`RegionIoU` computes the intersection-over-union between two sets of regions. Here: which detected **person**
boxes overlap which **face** boxes on the same poster? (Swap `"Person"` for `"Logo"` to find logos covering faces.)

In [ ]:
response, _ = run(queries.overlap_candidates("Person", dataset=DATASET, limit=5))
candidates = table(response)
candidates

In [ ]:
if candidates.empty:
    print("No poster with both a Person and a face box in this dataset.")
else:
    image_id = candidates["id"].iloc[0]
    response, _ = run(queries.object_face_overlap(image_id, "Person"))
    people, faces = table(response, 1), table(response, 2)
    iou = pd.DataFrame(response[-1]["RegionIoU"]["IoU"],
                       index=[f"Person (score {s:.2f})" for s in people.score],
                       columns=[f"face (joy {j:.2f})" for j in faces.joy])
    display(iou.style.format("{:.2f}").background_gradient(cmap="Blues", axis=None))

## Is the object-detector cut-off sensible?

`enrich.py` drops object proposals scoring below `--confidence-threshold` (default 0.5) *before* storing them,
so it writes `enrich_stats.json` with what the detector returned in total. The table shows how many proposals
fall in each score range and how many the cut-off kept. Re-run with a different threshold
(`python enrich.py --only object --replace --confidence-threshold 0.7`) and compare.

In [ ]:
path = "enrich_stats.json"
if not os.path.exists(path):
    print("Run enrich.py first - it writes", path)
else:
    stats = json.load(open(path))
    cut = stats["confidence_threshold"]
    detector = stats["detectors"].get("object")
    if not detector:
        print("The last enrich run did not process the object detector.")
    else:
        hist = pd.DataFrame({
            "score range": [f"{i / 10:.1f} - {(i + 1) / 10:.1f}" for i in range(10)],
            "proposals": detector["score_histogram"],
        })
        hist["kept"] = [n if (i + 1) / 10 > cut else 0 for i, n in enumerate(detector["score_histogram"])]
        hist["share"] = (hist.proposals / max(1, hist.proposals.sum()) * 40).round().astype(int).map(lambda n: "#" * n)
        print(f"threshold {cut}: stored {detector['kept']} of {detector['seen']} proposals "
              f"({detector['kept'] / max(1, detector['seen']):.0%}) over {detector['images']} images")
        display(hist)

## Try your own

* Edit the arguments above (`"golf"`, thresholds, ...) or write a new query function in `queries.py`.
* Every query is saved as JSON in `queries/` - paste one into the ApertureDB web UI's query editor
  (`docker compose --profile ui up -d` -> http://localhost:8088, login admin / admin).
* `python graph_viz.py node _Image --where gcs_object_stem=<name> --label _Image=gcs_object_stem --open`
  draws any poster from the command line.
* Narrow everything to one folder with `DATASET = "<bucket>/<prefix>"` in the first cell.